# Agent Evaluation


    1. Build Your Agent - use Langgraph / GoogleADK / CrewAI | llm=Gemini/gpt/claude
    2. Create evaluation dataset - mlflow supports a specific format
    3. Define scorers/metrics - agentic ai specific = 
        A. Tool Selection Evaluation = ToolCallCorrectness; ToolCallEfficiency, exact Match
        B. Overall Response Quality = Correctness, completeness, Guidelines, RelevancetoQuery
        C. For Conversational Applications: Multi Turn Efficiency = ConversationCompleteness, ConversationToolCallEfficiency, UserFrustration, 
    4. Run Evaluations

## 1. Agent Implementation

In [1]:
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

In [41]:
client = MultiServerMCPClient({"TredenceMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                              "transport":"stdio"}})


tools = await client.get_tools()


agent = create_agent("google_genai:gemini-2.5-flash",tools)
await agent.ainvoke({"messages":[{"role":'user',"content":"what is the weather in Delhi?"}]})

{'messages': [HumanMessage(content='what is the weather in Delhi?', additional_kwargs={}, response_metadata={}, id='e2375944-8eef-4cef-86a5-055d31c02170'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_weather', 'arguments': '{"city": "Delhi"}'}, '__gemini_function_call_thought_signatures__': {'ec0da7d1-96db-4eb0-8195-8078a0d3fe64': 'CvcBAQw51sfJ5CVQTRQfTRhG7ElwEbS1HB157Mdtz9CzAd1zj5wOSE04zbdbyJORFs8PnFohaylnigPnyApAtbkZ2vRCxu+6YHolmp1qG1s7PaEYxlrUtNaBUy9yVfie4A/FpRYosTWjaLTZanW1GdxgcMaVVOqTI87AjFkOR9/Vx+95sKss9I/Syh2zhQJBHnv1o3afdfYAMOXdSi+XfQJT4c30objO5W8IwK8EWIFal5IQ13dIQDoUqDoftF9/DemMtKZPhaVHWum8FgivcW/1s6E+tXHD7dP3kSSDvAGHgrXvGUDU11JN2gysJwceVjXMD/SHNp9swg=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e1ab1-41c8-73d2-bb31-788b4d23a7b9-0', tool_calls=[{'name': 'get_current_weather', 'args': {'city': 'Delhi'}, 'id': 'ec0da7d1-96db-4eb0-81

In [55]:
import nest_asyncio
nest_asyncio.apply()

async def predict_function(task:str):
    response = await agent.ainvoke({"messages":[{"role":'user',"content":task}]})
    return response

In [56]:
predict_function("what is weather in London today?")

<coroutine object predict_function at 0x7f8085e0d0e0>

## 2. Dataset for Agent Evaluation

In [57]:
eval_data = [{"inputs":{"task":"what is temperature in delhi today?"},
              "expectations":{"answer":"The weather in Delhi is haze with a temperature of 306.2 Kelvin."},
              "tags":{"topic":"weather"}},
              {"inputs":{"task":"what is temperature in Mumbai today?"},
              "expectations":{"answer":"The current temperature in Mumbai is 307.14 Kelvin."},
              "tags":{"topic":"weather"}},
              {"inputs":{"task":"what is temperature in London today?"},
              "expectations":{"answer":'The weather in London is broken clouds with a temperature of 276.07 Kelvin.'},
              "tags":{"topic":"weather"}},
              ]

## 3. Define Scorers

In [58]:
import mlflow
mlflow.set_tracking_uri("http://20.81.137.57:5000/")

mlflow.set_experiment("AgentEvaluation")

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1778563090372, experiment_id='2', last_update_time=1778563090372, lifecycle_stage='active', name='AgentEvaluation', tags={}, trace_location=None, workspace='default'>

In [59]:
from mlflow.genai import scorer
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency


@scorer
def exact_match(outputs,expectations):
    return outputs['messages'][-1].content==expectations['answer']

## 4. Run the Evaluations

In [60]:
results = mlflow.genai.evaluate(data=eval_data, predict_fn= predict_function,
                                scorers=[exact_match,ToolCallCorrectness(model="gemini:/gemini-2.5-flash"), ToolCallEfficiency(model="gemini:/gemini-2.5-flash")])

2026/05/12 11:13:43 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
2026/05/12 11:13:44 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8086711e00> was created in a different Context
2026/05/12 11:13:44 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f80a4e70180> was created in a different Context
2026/05/12 11:13:45 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f80867c2240> was created in a different Context
2026/05/12 11:13:45 WARNING mlflow.utils.autologging_utils

Evaluating:   0%|          | 0/3 [Elapsed: 00:00, Remaining: ?]

2026/05/12 11:13:49 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8085ee5c80> was created in a different Context
2026/05/12 11:13:49 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8085e9e5c0> was created in a different Context
2026/05/12 11:13:49 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8085e80480> was created in a different Context
2026/05/12 11:13:49 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f81119eb4c0> at 0x7f8057db3000> was created in a different Context
2026/05/12 11:13:49 WARNING mlflow.utils.aut

In [67]:
%%writefile code23agentevaluation.py

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
import mlflow
from mlflow.genai import scorer
import asyncio
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency
from dotenv import load_dotenv
load_dotenv()


mlflow.set_tracking_uri("http://20.81.137.57:5000/")
mlflow.set_experiment("AgentEvaluation")


async def main():

    client = MultiServerMCPClient({"TredenceMCP":{"command":"python","args":["mcpserver/mcpserver2.py"],
                                                "transport":"stdio"}})
        
    tools = await client.get_tools()
    agent = create_agent("google_genai:gemini-2.5-flash",tools)

    async def predict_function(task:str):
        response = await agent.ainvoke({"messages":[{"role":'user',"content":task}]})
        return response

    eval_data = [{"inputs":{"task":"what is temperature in delhi today?"},
                "expectations":{"answer":"The weather in Delhi is haze with a temperature of 306.2 Kelvin."},
                "tags":{"topic":"weather"}},
                {"inputs":{"task":"what is temperature in Mumbai today?"},
                "expectations":{"answer":"The current temperature in Mumbai is 307.14 Kelvin."},
                "tags":{"topic":"weather"}},
                {"inputs":{"task":"what is temperature in London today?"},
                "expectations":{"answer":'The weather in London is broken clouds with a temperature of 276.07 Kelvin.'},
                "tags":{"topic":"weather"}},
                ]


    @scorer
    def exact_match(outputs,expectations):
        return outputs['messages'][-1].content==expectations['answer']


    results = mlflow.genai.evaluate(data=eval_data, predict_fn= predict_function,
                                    scorers=[exact_match,ToolCallCorrectness(model="gemini:/gemini-2.5-flash"), ToolCallEfficiency(model="gemini:/gemini-2.5-flash")])

if __name__=="__main__":
    asyncio.run(main())

Overwriting code23agentevaluation.py


In [69]:
%%writefile code23agentevaluation.py

from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
import mlflow
from mlflow.genai import scorer
import asyncio
from mlflow.genai.scorers import ToolCallCorrectness, ToolCallEfficiency
from dotenv import load_dotenv
from langchain.tools import tool
import requests,json,wikipedia
load_dotenv()


mlflow.set_tracking_uri("http://20.81.137.57:5000/")
mlflow.set_experiment("AgentEvaluation")


tool()
def get_current_weather(city:str)->dict:
    """ this funciton can be used to get current weather information"""
    api_key="6a8b0ac166a37e2b7a38e64416b3c3fe"
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
    response = requests.get(url)
    response = json.loads(response.content.decode())
    output = {"city":city,"weather":response['weather'][0]['description'],
              "temperature":response['main']['temp'], "unit":"kelvin"
              }
    return output

tool()
def get_wikipedia_summary(query:str)->str:
    "get informaiton about any places, people, events from wikipedia"
    response = wikipedia.summary(query)
    return response


tools = [get_current_weather,get_wikipedia_summary]

def main():

    agent = create_agent("google_genai:gemini-2.5-flash",tools)

    def predict_function(task:str):
        response = agent.invoke({"messages":[{"role":'user',"content":task}]})
        return response

    eval_data = [{"inputs":{"task":"what is temperature in delhi today?"},
                "expectations":{"answer":"The weather in Delhi is haze with a temperature of 306.2 Kelvin."},
                "tags":{"topic":"weather"}},
                {"inputs":{"task":"what is temperature in Mumbai today?"},
                "expectations":{"answer":"The current temperature in Mumbai is 307.14 Kelvin."},
                "tags":{"topic":"weather"}},
                {"inputs":{"task":"what is temperature in London today?"},
                "expectations":{"answer":'The weather in London is broken clouds with a temperature of 276.07 Kelvin.'},
                "tags":{"topic":"weather"}},
                ]


    @scorer
    def exact_match(outputs,expectations):
        return outputs['messages'][-1].content==expectations['answer']


    results = mlflow.genai.evaluate(data=eval_data, predict_fn= predict_function,
                                    scorers=[exact_match,ToolCallCorrectness(model="gemini:/gemini-2.5-flash"), ToolCallEfficiency(model="gemini:/gemini-2.5-flash")])

if __name__=="__main__":
    main()

Overwriting code23agentevaluation.py


In [73]:
import os
os.environ['MLFLOW_GENAI_EVAL_ASYNC_TIMEOUT']="600"